In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

c:\Users\siwen\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Declaring the Quen3-Coder-Next Model and Loading the Tokenizer

In [2]:
model_name = "distilgpt2"

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model loaded successfully!")

c:\Users\siwen\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\siwen\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 380.83it/s, Materializing param=transformer.wte.weight]            


Model loaded successfully!


README Generation

In [9]:
def generate_readme(parsed_file, model, tokenizer):
    # Check if the parsed file exists, and if so it reads it
    if not os.path.exists(parsed_file):
        print(f"File {parsed_file} does not exist.")
        return None

    with open(parsed_file, 'r', encoding='utf-8', errors='ignore') as f:
        code_content = f.read()

    # Create a prompt for the model to generate the README.md content based on the code structure and contents
    prompt = f"Generate a comprehensive README.md file based on the following repository code structure and contents:\n\n{code_content}\n\n"
    
    # Set pad token if not already set
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Tokenize with truncation
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    # Generate the README.md content using the model
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
    
    # Decode the generated output and extract the README.md content
    readme_content = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "README.md" in readme_content:
        readme_content = readme_content.split("README.md")[1].strip()
        
    return readme_content

Saving the README in llm_output directory

In [10]:
def save_readme(output_path, readme_content, prefix=""):
    # Saves the generated README.md content to a file in the specified output path
    filename = f"{prefix}generated_README.md" if prefix else "generated_README.md"
    output_path = os.path.join(output_path, filename)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(readme_content)

    print(f"{prefix}README.md has been saved to {output_path}")
    return output_path

Generating the README from a repository

In [11]:
parsed_file = "out\\apache_cordova-plugin-splashscreen_parsed_code.txt"
output_path = "llm_output"

readme = generate_readme(parsed_file, model, tokenizer)
save_readme(output_path, readme)

README.md has been saved to llm_output\generated_README.md


'llm_output\\generated_README.md'